In [1]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-veksge_d
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-veksge_d
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=2d53cd0e6f09707fdefa35e7f8de7f9861ad25c854a6fde57bd6268b742b152d
  Stored in directory: /tmp/pip-ephem-wheel-cache-mw86bf4g/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [2]:
import os
import torch
import clip
import numpy as np
import pandas as pd
import html

from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "/kaggle/input/datasets/gianmarco96/upmcfood101"
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "images/train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "images/test")

TRAIN_TEXT = os.path.join(DATA_DIR, "texts/train_titles.csv")
TEST_TEXT = os.path.join(DATA_DIR, "texts/test_titles.csv")

model, preprocess = clip.load("ViT-B/32", device=DEVICE)

model = model.float()

100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 131MiB/s]


In [4]:
train_df = pd.read_csv(TRAIN_TEXT, header=None)
test_df = pd.read_csv(TEST_TEXT, header=None)

train_df.columns = ["image_name", "text", "class"]
test_df.columns = ["image_name", "text", "class"]

print(train_df.head())

          image_name                                               text  \
0  apple_pie_851.jpg    Crock-Pot Ladies  Crock-Pot Apple Pie Moonshine   
1  apple_pie_140.jpg       Mom's Maple-Apple Pie Recipe | Taste of Home   
2  apple_pie_858.jpg  Cookin&#8217; Canuck &#8211; Baked Apple Pie E...   
3  apple_pie_449.jpg      Dutch Apple Pie Recipe | Just A Pinch Recipes   
4  apple_pie_695.jpg  Our Share of the Harvest &raquo; Grandma&#8217...   

       class  
0  apple_pie  
1  apple_pie  
2  apple_pie  
3  apple_pie  
4  apple_pie  


In [5]:
def build_text_map(df):
    mapping = {}

    for _, row in df.iterrows():
        img = row["image_name"]
        text = html.unescape(str(row["text"]))
        cls = row["class"]

        # enrich text
        full_text = f"{text}. This is {cls}"

        mapping[img] = full_text

    return mapping


train_text_map = build_text_map(train_df)
test_text_map = build_text_map(test_df)

print("Sample:", list(train_text_map.items())[:2])

Sample: [('apple_pie_851.jpg', 'Crock-Pot Ladies  Crock-Pot Apple Pie Moonshine. This is apple_pie'), ('apple_pie_140.jpg', "Mom's Maple-Apple Pie Recipe | Taste of Home. This is apple_pie")]


In [6]:
def load_image_paths(root_dir):
    image_paths = []
    labels = []
    class_names = sorted(os.listdir(root_dir))

    class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

    for cls in class_names:
        cls_folder = os.path.join(root_dir, cls)
        for img_name in os.listdir(cls_folder):
            image_paths.append(os.path.join(cls_folder, img_name))
            labels.append(class_to_idx[cls])

    return image_paths, labels, class_names


train_paths, train_labels, class_names = load_image_paths(TRAIN_IMG_DIR)
test_paths, test_labels, _ = load_image_paths(TEST_IMG_DIR)

print("Classes:", len(class_names))

Classes: 101


In [8]:
def multimodal_zero_shot(test_paths, class_names, text_map):
    prompts = [f"a photo of {c}" for c in class_names]
    text_tokens = clip.tokenize(prompts).to(DEVICE)

    with torch.no_grad():
        class_features = model.encode_text(text_tokens)
        class_features /= class_features.norm(dim=-1, keepdim=True)

    preds = []

    for path in tqdm(test_paths):
        img_name = os.path.basename(path)
        real_text = text_map.get(img_name, "")

        image = preprocess(Image.open(path)).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            image_feat = model.encode_image(image)
            image_feat /= image_feat.norm(dim=-1, keepdim=True)

            text_tokens = clip.tokenize([real_text], truncate=True).to(DEVICE)
            
            text_feat = model.encode_text(text_tokens)
            text_feat /= text_feat.norm(dim=-1, keepdim=True)

            sim_class = image_feat @ class_features.T
            sim_text = image_feat @ text_feat.T

            combined = sim_class + 0.3 * sim_text
            pred = combined.argmax().item()

        preds.append(pred)

    return preds

zs_preds = multimodal_zero_shot(test_paths, class_names, test_text_map)
zs_acc = accuracy_score(test_labels, zs_preds)

print(f"Multimodal Zero-shot Accuracy: {zs_acc:.4f}")

100%|██████████| 22716/22716 [12:27<00:00, 30.40it/s]

Multimodal Zero-shot Accuracy: 0.5908


In [9]:
from collections import defaultdict
import random

def sample_k_shot(paths, labels, k=5):
    data = defaultdict(list)

    for p, l in zip(paths, labels):
        data[l].append(p)

    new_paths = []
    new_labels = []

    for l, items in data.items():
        sampled = random.sample(items, min(k, len(items)))
        new_paths.extend(sampled)
        new_labels.extend([l]*len(sampled))

    return new_paths, new_labels


few_train_paths, few_train_labels = sample_k_shot(train_paths, train_labels, k=5)

print("Few-shot samples:", len(few_train_paths))

Few-shot samples: 505


In [15]:
def extract_multimodal_features(paths, text_map):
    feats = []

    for path in tqdm(paths):
        img_name = os.path.basename(path)
        text = text_map.get(img_name, "")

        image = preprocess(Image.open(path)).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            img_feat = model.encode_image(image)
            img_feat /= img_feat.norm(dim=-1, keepdim=True)

            txt_feat = model.encode_text(clip.tokenize([text], truncate=True).to(DEVICE))
            txt_feat /= txt_feat.norm(dim=-1, keepdim=True)

        combined = torch.cat([img_feat, txt_feat], dim=-1)
        feats.append(combined.cpu().numpy()[0])

    return np.array(feats)

In [17]:
print("Extract train features...")
X_train = extract_multimodal_features(few_train_paths, train_text_map)

print("Extract test features...")
X_test = extract_multimodal_features(test_paths, test_text_map)

print("Training classifier...")
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, few_train_labels)

fs_preds = clf.predict(X_test)
fs_acc = accuracy_score(test_labels, fs_preds)

print(f"Multimodal Few-shot Accuracy: {fs_acc:.4f}")

Extract train features...


100%|██████████| 505/505 [00:12<00:00, 42.03it/s]


Extract test features...


100%|██████████| 22716/22716 [08:51<00:00, 42.70it/s]


Training classifier...
Multimodal Few-shot Accuracy: 0.9671
